# CSC2042S Assignment 1: Unsupervised learning

## 1. Data Preprocessing
Loading the **WDI** dataset and preparing it for clustering.

In [ ]:
import pandas as pd # TODO: Move all imports and constant defines to different cells for clarity
import numpy as np


# from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
# from sklearn.cluster import KMeans
from sklearn.manifold import TSNE

import matplotlib.pyplot as plt
# ensure plots display in notebook
%matplotlib inline

RANDOM_STATE: int = 42
WDI_CSV_PATH: str = 'WDICSV.csv'

pd.set_option('display.max_columns', 120)
rawDf = pd.read_csv(WDI_CSV_PATH)

yearCols = [c for c in rawDf.columns if c.isdigit()]
dataCols = ['Country Name','Country Code','Indicator Name','Indicator Code']

longDf = rawDf.melt(id_vars = dataCols, value_vars = yearCols, var_name = 'Year', value_name = 'Value')
longDf['Year'] = pd.to_numeric(longDf['Year'], errors = 'coerce').astype('Int64')
longDf['Value'] = pd.to_numeric(longDf['Value'], errors = 'coerce')

df = longDf.pivot_table(index=['Country Name','Country Code','Year'],
                       columns='Indicator Code',
                       values='Value',
                       aggfunc='mean').sort_index()
df.head()


### Filtering the data to remove some nulls
TODO: Actually document this properly, cos rn idk what the heck is going on here

In [ ]:
from typing import Any

ROW_THRESH: float = 0.7
FEAT_THRESH: float = 0.7  # Adjust thresholds as needed

def ProcessMissing(df) -> dict[str, Any]:
    total = df.size
    missing = df.isna().sum().sum()
    return {
        'rows': df.shape[0],
        'features': df.shape[1],
        'missing_pct': float(missing/total) if total > 0 else np.nan
    }

filtered = df.copy()
featCov = filtered.notna().mean(axis=0)
filtered = filtered.loc[:, featCov >= FEAT_THRESH]
rowCov = filtered.notna().mean(axis=1)
filtered = filtered.loc[rowCov >= ROW_THRESH]
ProcessMissing(filtered)

### Impute missing values (whatever that means)

In [ ]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy="mean")

x = imputer.fit_transform(filtered.iloc[:, 5:])  # assuming first cols are metadata
print(x)

### Scale everything zero to one

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
xScaled = scaler.fit_transform(x)
print(xScaled)

### Dimensionality reduction for visualization

In [ ]:
xEmbedded = TSNE(n_components=2, random_state=42).fit_transform(xScaled)

plt.figure(figsize=(8,6))
plt.scatter(xEmbedded[:,0], xEmbedded[:,1], s = 5, alpha = 0.5) # What the heck is [:,1] syntax in Python
plt.title("t-SNE Visualization of WDI Data")
plt.show()

### K-Means Clustering

In [ ]:
import numpy as np

def EuclideanDistance(a, b):
    return np.linalg.norm(a - b) # a and b should be vectors here

def InitializeCentroids(x, k):
    indices = np.random.choice(x.shape[0], k, replace = False)
    return x[indices]

def AssignClusters(x, centroids):
    clusters = []
    for myX in x:
        distances = [EuclideanDistance(myX, c) for c in centroids]
        clusterIndex = np.argmin(distances) # Find the index of the shortest distance
        clusters.append(clusterIndex)
        
    return np.array(clusters)

def UpdateCentroids(x, clusters, k):
    centroids = []
    for i in range(k):
        points = x[clusters == i] # What the heck is going on here with the '=='
        if len(points) > 0:
            centroids.append(points.mean(axis = 0))
        else:
            centroids.append(x[np.random.randint(0, x.shape[0])])

    return np.array(centroids)

def KMeans(x, k, maxIters = 100, tol = 1e-4):
    centroids = InitializeCentroids(x, k)
    
    for _ in range(maxIters):
        clusters = AssignClusters(x, centroids)
        newCentroids = UpdateCentroids(x, clusters, k)
        
        if np.all(np.abs(newCentroids - centroids) < tol):
            break
        
        centroids = newCentroids
    
    return clusters, centroids

### Performing the clustering and visualization

In [ ]:
k = 16 # TODO: We need to find the optimal k using the elbow & silhouette method
clusters, centroids = KMeans(xScaled, k)

print("Cluster assignments:", clusters)
print("Centroids:", centroids)

# Plot
plt.figure(figsize=(8,6))
plt.scatter(
    xEmbedded[:,0],   # x-axis
    xEmbedded[:,1],   # y-axis
    c=clusters,       # color by cluster
    cmap="tab20",     # 20 distinct colors (works well since k=16)
    s=10,             # point size
    alpha=0.7         # transparency
)
plt.title(f"t-SNE Visualization of Clusters (k={k})")
plt.colorbar(label = "Cluster ID")
plt.show()